# 19 — Transformer Encoder in PyTorch

**Learning objective.** Assemble embeddings, positional information, multi-head self-attention and feed-forward blocks using torch.nn.

This notebook is intentionally **offline-reproducible**: the examples use local data or deterministic toy corpora so the rendered GitHub output can be trusted without hidden API calls. The focus is always **concept → inspectable representation → library implementation → result → failure modes → production implication**.

In [1]:
from pathlib import Path
import re, math, json, random, statistics
import numpy as np
import pandas as pd
np.random.seed(42)
random.seed(42)
pd.set_option('display.max_colwidth', 120)
DATA = Path('data')
print('Reproducibility seed: 42')

Reproducibility seed: 42


In [2]:
import torch, torch.nn as nn
torch.manual_seed(42)
seq_len,batch,d_model=5,2,16
x=torch.randn(batch,seq_len,d_model)
pos=nn.Parameter(torch.randn(1,seq_len,d_model))
layer=nn.TransformerEncoderLayer(d_model=d_model,nhead=4,dim_feedforward=32,batch_first=True,dropout=0.0)
encoder=nn.TransformerEncoder(layer,num_layers=2)
y=encoder(x+pos)
print('input :',tuple(x.shape))
print('output:',tuple(y.shape))
print('same sequence length and model width:',x.shape==y.shape)

input : (2, 5, 16)
output: (2, 5, 16)
same sequence length and model width: True


### What the encoder block adds
1. **Multi-head self-attention** mixes information across token positions.
2. **Feed-forward network** transforms each position independently.
3. **Residual connections + normalization** stabilize deep optimization.
4. **Position information** breaks the permutation symmetry of pure attention.

In [3]:
print(layer)

TransformerEncoderLayer(
  (self_attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=16, out_features=16, bias=True)
  )
  (linear1): Linear(in_features=16, out_features=32, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
  (linear2): Linear(in_features=32, out_features=16, bias=True)
  (norm1): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
  (norm2): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
  (dropout1): Dropout(p=0.0, inplace=False)
  (dropout2): Dropout(p=0.0, inplace=False)
)


---
    ## Production takeaways
    - Preserve preprocessing as part of the model contract; training/inference skew is an NLP failure mode, not an implementation detail.
    - Inspect intermediate representations rather than treating tokenizers/vectorizers/models as black boxes.
    - Prefer the simplest representation/model that meets quality, latency, governance, and maintenance requirements.

### What you should now be able to explain
- Trace transformer tensor shapes
- Explain why positional information is necessary